# 🎮 Sysadmin Game: GRPO Training with Live Environment

This notebook trains a Qwen2.5-Coder model using **Group Relative Policy Optimization (GRPO)** with live Docker container feedback.

## Architecture

```
┌─────────────────────────────────┐
│  Colab (GPU)                    │
│  - Model inference/training     │
│  - GRPO optimization            │
└───────────────┬─────────────────┘
                │ HTTP
┌───────────────▼─────────────────┐
│  Your Machine (Docker)          │
│  - OpenEnv server               │
│  - Sandbox containers           │
│  - Exposed via ngrok            │
└─────────────────────────────────┘
```

**Requirements**: 
- GPU runtime in Colab (T4 minimum)
- Environment server running on your machine (see setup below)

---

## 0. Start Environment Server (Run on YOUR Machine)

Before running this notebook, start the environment server locally:

```bash
# Terminal 1: Start the server
cd sysadmin
uv run sysadmin-server

# Terminal 2: Expose via ngrok
ngrok http 8000
```

Copy the ngrok URL (e.g., `https://abc123.ngrok.io`) and paste it below.

In [ ]:
#@title 🔗 Enter your ngrok URL
ENV_URL = "https://YOUR-NGROK-URL.ngrok.io"  #@param {type:"string"}

# Verify connection
import requests

try:
    r = requests.get(f"{ENV_URL}/health", timeout=10)
    if r.status_code == 200:
        print(f"✅ Connected to environment server!")
        print(f"   Status: {r.json()}")
    else:
        print(f"❌ Server returned status {r.status_code}")
except Exception as e:
    print(f"❌ Could not connect to {ENV_URL}")
    print(f"   Error: {e}")
    print("\n👉 Make sure your server is running and ngrok URL is correct!")

## 1. Setup Colab Environment

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install Unsloth (optimized for Colab)
%%capture
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
# Install other dependencies
%%capture
!pip install trl>=0.8.0 transformers>=4.40.0 datasets>=2.18.0 wandb>=0.16.0 matplotlib peft bitsandbytes requests

## 2. Configuration

In [ ]:
# GRPO Training Configuration
CONFIG = {
    # Environment server URL (from ngrok)
    "env_url": ENV_URL,

    # Model - use 3B for T4, 7B for A100
    "model_name": "Qwen/Qwen2.5-Coder-3B-Instruct",

    # Start from SFT checkpoint if available
    "sft_checkpoint": None,  # Set path if you ran SFT first

    # LoRA settings
    "lora_r": 16,
    "lora_alpha": 16,

    # GRPO settings
    "num_steps": 100,           # Training steps
    "episodes_per_step": 8,     # Episodes to collect per step
    "group_size": 4,            # GRPO group size
    "max_turns": 10,            # Max commands per episode
    "learning_rate": 1e-5,

    # Generation
    "max_seq_length": 2048,
    "temperature": 0.7,

    # Paths
    "output_dir": "checkpoints/grpo",

    # Logging
    "use_wandb": False,
    "wandb_project": "sysadmin-game",
    "save_every": 25,
}

print("Configuration loaded!")
print(f"Environment: {CONFIG['env_url']}")
print(f"Model: {CONFIG['model_name']}")
print(f"Training steps: {CONFIG['num_steps']}")

In [ ]:
# Optional: Login to W&B
if CONFIG["use_wandb"]:
    import wandb
    wandb.login()
    wandb.init(project=CONFIG["wandb_project"], name="grpo-training", config=CONFIG)

## 3. Load Model

In [ ]:
from unsloth import FastLanguageModel
import torch

model_path = CONFIG["sft_checkpoint"] or CONFIG["model_name"]
print(f"Loading model: {model_path}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_path,
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,
    load_in_4bit=True,
)

# Add LoRA adapters if loading base model
if CONFIG["sft_checkpoint"] is None:
    print("Adding LoRA adapters to base model")
    model = FastLanguageModel.get_peft_model(
        model,
        r=CONFIG["lora_r"],
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_alpha=CONFIG["lora_alpha"],
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
    )

print(f"Model loaded! Device: {model.device}")

## 4. OpenEnv HTTP Client

In [ ]:
import re
import requests
from dataclasses import dataclass
from typing import Optional


@dataclass
class Observation:
    """Environment observation."""
    output: str
    done: bool
    reward: float
    metadata: dict


class OpenEnvClient:
    """HTTP client for the OpenEnv sysadmin server."""

    def __init__(self, base_url: str):
        self.base_url = base_url.rstrip("/")
        self.session = requests.Session()

    def health(self) -> dict:
        """Check server health."""
        r = self.session.get(f"{self.base_url}/health")
        r.raise_for_status()
        return r.json()

    def reset(self, scenario_id: Optional[str] = None, seed: Optional[int] = None) -> Observation:
        """Reset to a new episode."""
        payload = {}
        if scenario_id:
            payload["scenario_id"] = scenario_id
        if seed is not None:
            payload["seed"] = seed

        r = self.session.post(f"{self.base_url}/reset", json=payload)
        r.raise_for_status()
        data = r.json()
        return Observation(**data)

    def step(self, command: str) -> Observation:
        """Execute a command."""
        r = self.session.post(f"{self.base_url}/step", json={"command": command})
        r.raise_for_status()
        data = r.json()
        return Observation(**data)

    def scenarios(self) -> dict:
        """Get available scenarios."""
        r = self.session.get(f"{self.base_url}/scenarios")
        r.raise_for_status()
        return r.json()


# System prompt
SYSTEM_PROMPT = """You are an expert SRE agent that diagnoses and fixes Linux system issues.

When given a problem:
1. Think step-by-step about the diagnosis in <think> tags
2. Run ONE command at a time in <bash> tags
3. Analyze the output before running the next command
4. Continue until the issue is fixed

Example response:
<think>
The user reports nginx won't start. First, let me check the service status.
</think>
<bash>systemctl status nginx --no-pager -l</bash>"""


def parse_response(response: str) -> tuple[str | None, str | None]:
    """Extract thinking and bash command."""
    think_match = re.search(r"<think>(.*?)</think>", response, re.DOTALL)
    bash_match = re.search(r"<bash>(.*?)</bash>", response, re.DOTALL)
    thinking = think_match.group(1).strip() if think_match else None
    command = bash_match.group(1).strip() if bash_match else None
    return thinking, command


print("OpenEnv client defined!")

In [ ]:
# Test connection and run a single episode
client = OpenEnvClient(CONFIG["env_url"])

print("Testing environment connection...")
print(f"Health: {client.health()}")
print(f"Scenarios: {client.scenarios()['train'][:5]}...")

# Test reset/step
obs = client.reset()
print(f"\nReset OK! Scenario: {obs.metadata['scenario_id']}")
print(f"Complaint: {obs.output[:100]}...")

obs = client.step("df -h")
print(f"\nStep OK! Reward: {obs.reward}")

## 5. Episode Runner

In [ ]:
def run_episode(client: OpenEnvClient, model, tokenizer, max_turns: int) -> dict:
    """Run a single episode via HTTP."""
    obs = client.reset()

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": obs.output},
    ]

    trajectory = {
        "scenario_id": obs.metadata["scenario_id"],
        "prompts": [],
        "responses": [],
        "rewards": [],
        "commands": [],
    }

    total_reward = 0.0

    for turn in range(max_turns):
        # Format prompt
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

        # Generate response
        FastLanguageModel.for_inference(model)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                temperature=CONFIG["temperature"],
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )

        response = tokenizer.decode(
            outputs[0][inputs.input_ids.shape[1]:],
            skip_special_tokens=True
        )

        # Parse response
        thinking, command = parse_response(response)

        trajectory["prompts"].append(prompt)
        trajectory["responses"].append(response)

        if not command:
            trajectory["rewards"].append(-0.1)
            trajectory["commands"].append(None)
            total_reward -= 0.1
            break

        trajectory["commands"].append(command)

        # Execute via HTTP
        obs = client.step(command)
        trajectory["rewards"].append(obs.reward)
        total_reward += obs.reward

        messages.append({"role": "assistant", "content": response})
        messages.append({"role": "tool", "content": f"<output>\n{obs.output}\n</output>"})

        if obs.done:
            break

    trajectory["total_reward"] = total_reward
    trajectory["fixed"] = obs.metadata.get("fixed", False)
    trajectory["num_commands"] = len([c for c in trajectory["commands"] if c])

    return trajectory


# Test single episode
print("Testing single episode...")
traj = run_episode(client, model, tokenizer, max_turns=5)
print(f"Scenario: {traj['scenario_id']}")
print(f"Commands: {traj['num_commands']}")
print(f"Reward: {traj['total_reward']:.3f}")
print(f"Fixed: {traj['fixed']}")

## 6. GRPO Training Loop

In [ ]:
import os
from collections import defaultdict

os.makedirs(CONFIG["output_dir"], exist_ok=True)
os.makedirs("results", exist_ok=True)


def collect_trajectories(client, model, tokenizer, num_episodes: int) -> list[dict]:
    """Collect episodes via HTTP."""
    trajectories = []
    for i in range(num_episodes):
        try:
            traj = run_episode(client, model, tokenizer, CONFIG["max_turns"])
            trajectories.append(traj)
        except Exception as e:
            print(f"    Episode {i+1} failed: {e}")
    return trajectories


def compute_grpo_advantages(trajectories: list[dict], group_size: int) -> list[dict]:
    """Compute GRPO advantages."""
    if len(trajectories) < group_size:
        return []

    processed = []
    for i in range(0, len(trajectories) - group_size + 1, group_size):
        group = trajectories[i:i + group_size]
        rewards = [t["total_reward"] for t in group]

        mean_reward = sum(rewards) / len(rewards)
        std_reward = (sum((r - mean_reward) ** 2 for r in rewards) / len(rewards)) ** 0.5
        std_reward = max(std_reward, 1e-6)

        for traj in group:
            advantage = (traj["total_reward"] - mean_reward) / std_reward
            for prompt, response, reward in zip(
                traj["prompts"], traj["responses"], traj["rewards"]
            ):
                processed.append({
                    "prompt": prompt,
                    "response": response,
                    "reward": reward,
                    "advantage": advantage,
                })
    return processed

In [ ]:
# Training metrics
history = {"steps": [], "rewards": [], "fix_rates": [], "avg_commands": [], "losses": []}
scenario_stats = defaultdict(lambda: {"attempts": 0, "fixes": 0})

print(f"Starting GRPO training: {CONFIG['num_steps']} steps")
print(f"Episodes per step: {CONFIG['episodes_per_step']}")
print("="*60)

In [ ]:
# Main training loop
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"])

for step in range(CONFIG["num_steps"]):
    print(f"\nStep {step + 1}/{CONFIG['num_steps']}")

    # Collect trajectories via HTTP
    trajectories = collect_trajectories(
        client, model, tokenizer,
        num_episodes=CONFIG["episodes_per_step"]
    )

    if not trajectories:
        print("  No trajectories, skipping")
        continue

    # Metrics
    rewards = [t["total_reward"] for t in trajectories]
    fix_rate = sum(1 for t in trajectories if t["fixed"]) / len(trajectories)
    avg_reward = sum(rewards) / len(rewards)
    avg_commands = sum(t["num_commands"] for t in trajectories) / len(trajectories)

    for t in trajectories:
        scenario_stats[t["scenario_id"]]["attempts"] += 1
        if t["fixed"]:
            scenario_stats[t["scenario_id"]]["fixes"] += 1

    print(f"  Reward: {avg_reward:.3f} | Fix: {fix_rate:.1%} | Cmds: {avg_commands:.1f}")

    # GRPO update
    training_data = compute_grpo_advantages(trajectories, CONFIG["group_size"])
    if not training_data:
        continue

    FastLanguageModel.for_training(model)
    model.train()

    total_loss = 0.0
    for example in training_data:
        inputs = tokenizer(
            example["prompt"] + example["response"],
            return_tensors="pt",
            truncation=True,
            max_length=CONFIG["max_seq_length"],
        ).to(model.device)

        outputs = model(**inputs, labels=inputs.input_ids)
        loss = outputs.loss * example["advantage"]

        if not (torch.isnan(loss) or torch.isinf(loss)):
            loss.backward()
            total_loss += loss.item()

    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    optimizer.zero_grad()

    avg_loss = total_loss / max(len(training_data), 1)
    print(f"  Loss: {avg_loss:.4f}")

    # Store
    history["steps"].append(step + 1)
    history["rewards"].append(avg_reward)
    history["fix_rates"].append(fix_rate)
    history["avg_commands"].append(avg_commands)
    history["losses"].append(avg_loss)

    if CONFIG["use_wandb"]:
        wandb.log({"reward": avg_reward, "fix_rate": fix_rate, "loss": avg_loss})

    # Checkpoint
    if (step + 1) % CONFIG["save_every"] == 0:
        path = f"{CONFIG['output_dir']}/step_{step + 1}"
        model.save_pretrained(path)
        tokenizer.save_pretrained(path)
        print(f"  Saved: {path}")

print("\n" + "="*60)
print("GRPO training complete!")

In [ ]:
# Save final model
model.save_pretrained(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])
print(f"Final model saved to {CONFIG['output_dir']}")

## 7. Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].plot(history["steps"], history["rewards"], 'b-', lw=2)
axes[0,0].set_xlabel("Step"); axes[0,0].set_ylabel("Reward")
axes[0,0].set_title("Average Reward"); axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(history["steps"], history["fix_rates"], 'g-', lw=2)
axes[0,1].set_xlabel("Step"); axes[0,1].set_ylabel("Fix Rate")
axes[0,1].set_title("Success Rate"); axes[0,1].set_ylim(0,1); axes[0,1].grid(True, alpha=0.3)

axes[1,0].plot(history["steps"], history["avg_commands"], 'orange', lw=2)
axes[1,0].set_xlabel("Step"); axes[1,0].set_ylabel("Commands")
axes[1,0].set_title("Avg Commands to Fix"); axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(history["steps"], history["losses"], 'r-', lw=2)
axes[1,1].set_xlabel("Step"); axes[1,1].set_ylabel("Loss")
axes[1,1].set_title("Policy Loss"); axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("results/grpo_curves.png", dpi=150)
plt.show()

In [ ]:
# Per-scenario stats
print("Per-Scenario Performance:")
print("-" * 50)
for sid, stats in sorted(scenario_stats.items()):
    rate = stats["fixes"] / stats["attempts"] if stats["attempts"] > 0 else 0
    print(f"{sid:<20} {stats['attempts']:>5} attempts, {stats['fixes']:>5} fixes ({rate:.0%})")

## 8. Download Results

In [ ]:
# Save results
import json
with open("results/grpo_results.json", "w") as f:
    json.dump({"history": history, "scenarios": dict(scenario_stats)}, f)

!zip -r grpo_results.zip checkpoints/grpo results/
print("Download grpo_results.zip from file browser")